# 2. Cierre del numeral 8

Este notebook produce toda la evidencia que el examen pide en la sección 8.2, a
partir de los resultados que ya generó el Cuaderno14. **No carga modelos**: solo
lee `results/scores_por_caso.csv`, así que corre en segundos y sin GPU.

| Punto | Qué se produce |
|---|---|
| 8.2.2 / 8.5.3 | Tabla de efectos: tamaño vs contexto |
| 8.2.3 | Truncación por modelo y su relación con el rendimiento |
| 8.2.4 | Intervalos de confianza y prueba de estabilidad |
| 8.2.5 | Casos donde el título no basta |
| 8.5.1 | Auditoría de tokens desde `MODEL_SPECS` |
| 8.5.4 | Recall@K con varios captions correctos |

In [1]:
from pathlib import Path
from math import comb
import json
import sys
import numpy as np
import pandas as pd

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = BASE / "results"
MULTI = BASE / "multiedicion"
sys.path.insert(0, str(BASE))
from config_variante import VARIANTE, dir_salida

MODELOS = ["CLIP", "CLIP-L", "LongCLIP"]
CAPTIONS = ["caption_2", "caption_3", "caption_4"]
SEED, N_BOOT = 22514, 10000

MODEL_SPECS = {
    "CLIP":     {"checkpoint": "openai/clip-vit-base-patch32",
                 "max_tokens": 77,  "encoder_visual": "ViT-B/32", "patch": 32},
    "CLIP-L":   {"checkpoint": "openai/clip-vit-large-patch14",
                 "max_tokens": 77,  "encoder_visual": "ViT-L/14", "patch": 14},
    "LongCLIP": {"checkpoint": "zer0int/LongCLIP-GmP-ViT-L-14",
                 "max_tokens": 248, "encoder_visual": "ViT-L/14", "patch": 14},
}

# --- fuente de los rangos --------------------------------------------------
# Este notebook no abre imágenes: parte de los rangos que ya calculó otra
# corrida. Cuál se use cambia el significado de todas las tablas, así que se
# declara explícitamente y queda registrado en el nombre de la salida.
#
#   "multiedicion" las tres ediciones (n=95) en la variante de imágenes activa,
#                  producida por el notebook 03. Es la opción por defecto.
#   "cuaderno14"   los 40 gráficos de 2026-1 con título, corrida original del
#                  proyecto. Se conserva para poder reproducir el punto de
#                  partida y contrastar contra él.
FUENTE = "multiedicion"


def cargar_rangos(fuente=FUENTE, variante=VARIANTE):
    """Devuelve (tabla ancha de rangos, etiqueta, directorio de salida).

    La tabla ancha tiene una fila por gráfico y una columna
    rank_<modelo>_<caption>, que es el formato que usan las celdas siguientes.
    """
    if fuente == "multiedicion":
        f = (BASE / f"resultados_multiedicion_{variante}"
             / "scores_por_caso_multiedicion.csv")
        if not f.exists():
            raise FileNotFoundError(
                f"falta {f}\n\nEjecutar antes el notebook 03 con "
                f"VARIANTE='{variante}'. Si solo se quiere reproducir la corrida "
                f"original de 40 gráficos, poner FUENTE = 'cuaderno14'.")
        largo = pd.read_csv(f)
        # de formato largo (una fila por modelo y caption) a ancho
        casos = largo.pivot_table(index="image_id", columns=["modelo", "caption"],
                                  values="rank_i2t").reset_index()
        casos.columns = ["image_id" if c[0] == "image_id" else f"rank_{c[0]}_{c[1]}"
                         for c in casos.columns.to_flat_index()]
        ids = largo[["image_id", "chart_uid", "edicion"]].drop_duplicates()
        casos = casos.merge(ids, on="image_id", how="left")
        casos["chart_id"] = casos["chart_uid"].str.split("|").str[-1]
        etiqueta = (f"multiedición · 3 ediciones · imágenes {variante} · "
                    f"n={len(casos)}")
        salida = dir_salida(BASE, "resultados_cierre")

    elif fuente == "cuaderno14":
        f = RESULTS / "scores_por_caso.csv"
        if not f.exists():
            raise FileNotFoundError(f"falta {f}")
        casos = pd.read_csv(f)
        casos["edicion"] = "2026-1"
        etiqueta = "cuaderno14 · 2026-1 · con título · n=40"
        salida = dir_salida(BASE, "resultados_cierre_cuaderno14", variante=None) \
            if False else (BASE / "resultados_cierre_cuaderno14")
        salida.mkdir(exist_ok=True)

    else:
        raise ValueError("FUENTE debe ser 'multiedicion' o 'cuaderno14'")

    return casos, etiqueta, salida


casos, ETIQUETA, OUT = cargar_rangos()
disponibles = [c for c in casos.columns if c.startswith("rank_")]

print(f"fuente:  {ETIQUETA}")
print(f"salida:  {OUT}")
print(f"{len(casos)} gráficos · {len(disponibles)} combinaciones modelo×caption")
if "edicion" in casos.columns:
    print(casos.groupby("edicion").size().to_string())

fuente:  multiedición · 3 ediciones · imágenes con_titulo · n=95
salida:  /tf/work/final/sbs_iesf_pares/resultados_cierre_con_titulo
95 gráficos · 9 combinaciones modelo×caption
edicion
2021-1    26
2024-2    29
2026-1    40


## 2.1 Vector de aciertos

Todo lo que sigue se calcula sobre una matriz booleana: para cada gráfico y cada
combinación modelo×caption, si el caption correcto quedó en primer lugar.

Reconstruirlo desde los rangos permite tratar las comparaciones como **pareadas**:
los tres modelos vieron exactamente los mismos gráficos.

In [2]:
aciertos = pd.DataFrame({"image_id": casos["image_id"]})
for m in MODELOS:
    for c in CAPTIONS:
        aciertos[f"{m}|{c}"] = (casos[f"rank_{m}_{c}"] == 1).astype(int)

aciertos.head()

,image_id,CLIP|caption_2,CLIP|caption_3,CLIP|caption_4,CLIP-L|caption_2,CLIP-L|caption_3,CLIP-L|caption_4,LongCLIP|caption_2,LongCLIP|caption_3,LongCLIP|caption_4
0,2021-1_img_000,1,0,0,1,1,1,1,1,1
1,2021-1_img_001,0,0,1,1,1,1,0,1,1
2,2021-1_img_002,0,1,0,1,1,0,1,1,0
3,2021-1_img_003,1,1,1,1,1,1,0,1,1
4,2021-1_img_004,0,1,1,1,1,1,1,1,1


## 2.2 R@1 y MRR

El mapa del curso pide **R@K y MRR**, y las dos métricas responden preguntas
distintas.

R@K es binaria: ¿quedó el correcto entre los K primeros? MRR usa la posición
completa (1/rango promediado), de modo que premia dejar el correcto cerca del tope
aunque no gane.

En estos datos la diferencia no es teórica: hay condiciones que **empatan en R@1 y
se separan en MRR**.

In [3]:
filas = []
for m in MODELOS:
    for c in CAPTIONS:
        r = casos[f"rank_{m}_{c}"].astype(float)
        filas.append({"modelo": m, "caption": c,
                      "R@1": round((r == 1).mean(), 4),
                      "R@5": round((r <= 5).mean(), 4),
                      "MRR": round((1 / r).mean(), 4),
                      "rango_mediano": int(r.median()),
                      "rango_peor": int(r.max())})

mrr = pd.DataFrame(filas)
mrr.to_csv(OUT / "mrr.csv", index=False)
print(mrr.to_string(index=False))

print("\nEmpates en R@1 que MRR distingue:")
for m in MODELOS:
    g = mrr[mrr.modelo == m]
    for v in g["R@1"].unique():
        sub = g[g["R@1"] == v]
        if len(sub) > 1:
            det = ", ".join(f"{r.caption}={r.MRR}" for _, r in sub.iterrows())
            print(f"  {m}: R@1={v} -> MRR {det}")

  modelo   caption    R@1    R@5    MRR  rango_mediano  rango_peor
    CLIP caption_2 0.4000 0.7895 0.5505              2          33
    CLIP caption_3 0.4737 0.8000 0.6120              2          24
    CLIP caption_4 0.4842 0.7789 0.6160              2          34
  CLIP-L caption_2 0.6842 0.8526 0.7577              1          24
  CLIP-L caption_3 0.6316 0.9053 0.7437              1          25
  CLIP-L caption_4 0.6316 0.8632 0.7437              1          16
LongCLIP caption_2 0.6000 0.8000 0.6918              1          23
LongCLIP caption_3 0.6421 0.8947 0.7453              1          27
LongCLIP caption_4 0.6105 0.8105 0.7129              1          33

Empates en R@1 que MRR distingue:
  CLIP-L: R@1=0.6316 -> MRR caption_3=0.7437, caption_4=0.7437


## 2.3 Separar efecto de tamaño y efecto de contexto (8.2.2)

Esta es la pregunta central del proyecto. Comparar CLIP-B con LongCLIP no la
responde, porque entre esos dos modelos cambian **dos cosas a la vez**: el encoder
visual (ViT-B/32 → ViT-L/14) y el límite de tokens (77 → 248).

CLIP-L sirve de puente porque comparte una variable con cada extremo:

- **efecto de tamaño** = CLIP-L − CLIP-B → ambos con 77 tokens, cambia el encoder
- **efecto de contexto** = LongCLIP − CLIP-L → ambos con ViT-L/14, cambia la ventana

In [4]:
# el R@1 se recalcula desde los rangos de la fuente activa, en lugar de leer
# metricas.csv, que corresponde siempre al Cuaderno14
piv = pd.DataFrame({m: {cap: aciertos[f"{m}|{cap}"].mean() for cap in CAPTIONS}
                    for m in MODELOS})

efectos = pd.DataFrame([{
    "caption": c,
    "R@1_CLIP_B": piv.loc[c, "CLIP"],
    "R@1_CLIP_L": piv.loc[c, "CLIP-L"],
    "R@1_LongCLIP": piv.loc[c, "LongCLIP"],
    "efecto_tamano_L_menos_B": round(piv.loc[c, "CLIP-L"] - piv.loc[c, "CLIP"], 4),
    "efecto_contexto_Long_menos_L": round(piv.loc[c, "LongCLIP"] - piv.loc[c, "CLIP-L"], 4),
} for c in CAPTIONS])

efectos.to_csv(OUT / "tabla_efectos.csv", index=False)
print(efectos.to_string(index=False))

  caption  R@1_CLIP_B  R@1_CLIP_L  R@1_LongCLIP  efecto_tamano_L_menos_B  efecto_contexto_Long_menos_L
caption_2    0.400000    0.684211      0.600000                   0.2842                       -0.0842
caption_3    0.473684    0.631579      0.642105                   0.1579                        0.0105
caption_4    0.484211    0.631579      0.610526                   0.1474                       -0.0211


### Interpretación por tamaño de patch

El efecto de tamaño tiene una explicación concreta en un concepto del curso
(bloque *Transformers y VLM*: **patches**).

| Encoder | Patch | Parches en 224×224 |
|---|---|---|
| ViT-B/32 | 32×32 | 49 |
| ViT-L/14 | 14×14 | 256 |

ViT-L/14 divide la imagen en cinco veces más parches, es decir, con mucho mayor
detalle espacial. En un gráfico financiero la información distintiva está en
etiquetas de eje pequeñas y en la forma fina de una curva: un parche de 32 píxeles
puede promediar varias etiquetas en un solo token visual.

Esto convierte el hallazgo en algo más que una tabla de diferencias — y genera una
predicción verificable: la ventaja debería ser mayor en gráficos densos que en
gráficos simples.

In [5]:
for m, s in MODEL_SPECS.items():
    n = (224 // s["patch"]) ** 2
    print(f"{m:10s} {s['encoder_visual']:9s} patch {s['patch']:2d} -> {n:3d} parches"
          f" | {s['max_tokens']:3d} tokens")

CLIP       ViT-B/32  patch 32 ->  49 parches |  77 tokens
CLIP-L     ViT-L/14  patch 14 -> 256 parches |  77 tokens
LongCLIP   ViT-L/14  patch 14 -> 256 parches | 248 tokens


## 2.4 Intervalos de confianza (8.2.4)

Con 40 gráficos la incertidumbre es grande y hay que declararla. El bootstrap
remuestrea **imágenes** con reemplazo: cada réplica es un conjunto alternativo de
40 gráficos que pudo haber tocado, y la dispersión de los R@1 resultantes estima
cuánto depende el resultado de esta muestra particular.

In [6]:
rng = np.random.default_rng(SEED)
n = len(aciertos)
filas = []

for m in MODELOS:
    for c in CAPTIONS:
        v = aciertos[f"{m}|{c}"].to_numpy()
        idx = rng.integers(0, n, size=(N_BOOT, n))   # remuestreo con reemplazo
        boots = v[idx].mean(axis=1)
        lo, hi = np.percentile(boots, [2.5, 97.5])
        filas.append({"modelo": m, "caption": c, "R@1": round(v.mean(), 4),
                      "IC95_inf": round(lo, 4), "IC95_sup": round(hi, 4),
                      "amplitud_IC": round(hi - lo, 4), "n": n})

ic = pd.DataFrame(filas)
ic.to_csv(OUT / "intervalos_confianza.csv", index=False)
print(ic.to_string(index=False))
print(f"\namplitud media del IC95: {ic['amplitud_IC'].mean():.3f}")

  modelo   caption    R@1  IC95_inf  IC95_sup  amplitud_IC  n
    CLIP caption_2 0.4000    0.2947    0.5053       0.2105 95
    CLIP caption_3 0.4737    0.3789    0.5789       0.2000 95
    CLIP caption_4 0.4842    0.3895    0.5789       0.1895 95
  CLIP-L caption_2 0.6842    0.5895    0.7789       0.1895 95
  CLIP-L caption_3 0.6316    0.5263    0.7263       0.2000 95
  CLIP-L caption_4 0.6316    0.5368    0.7263       0.1895 95
LongCLIP caption_2 0.6000    0.5053    0.6947       0.1895 95
LongCLIP caption_3 0.6421    0.5474    0.7368       0.1895 95
LongCLIP caption_4 0.6105    0.5158    0.7053       0.1895 95

amplitud media del IC95: 0.194


## 2.5 Prueba de estabilidad (8.2.4)

Los tres modelos se evalúan sobre **los mismos** gráficos, así que la comparación
es pareada. Tratarla como dos proporciones independientes sería incorrecto: se
perdería la información de qué gráficos concretos cambian de resultado.

La prueba de McNemar usa exactamente esa información. Solo mira las
**discordancias**: los gráficos donde una condición acierta y la otra falla. Si el
cambio de modelo no aporta nada, esas discordancias deberían repartirse por mitades.

*Nota sobre el temario:* McNemar no figura en el mapa del curso; es el instrumento
para la **confiabilidad**, que sí figura en el bloque Evaluación. El resultado
principal reportado es el IC del punto anterior; esta prueba lo respalda.

In [7]:
def p_mcnemar(b, c):
    """p-valor exacto de dos colas. b y c son las discordancias."""
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    return min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / 2 ** n)


comparaciones = [
    (("CLIP", "caption_2"), ("CLIP-L", "caption_2")),        # tamaño
    (("CLIP-L", "caption_2"), ("LongCLIP", "caption_2")),    # contexto, cap corto
    (("CLIP-L", "caption_3"), ("LongCLIP", "caption_3")),    # contexto, cap largo
    (("CLIP", "caption_2"), ("LongCLIP", "caption_3")),      # acumulado
]

filas = []
for (m1, c1), (m2, c2) in comparaciones:
    a = aciertos[f"{m1}|{c1}"].to_numpy()
    d = aciertos[f"{m2}|{c2}"].to_numpy()
    b = int(((a == 1) & (d == 0)).sum())
    c = int(((a == 0) & (d == 1)).sum())
    p = p_mcnemar(b, c)
    filas.append({"condicion_1": f"{m1}|{c1}", "condicion_2": f"{m2}|{c2}",
                  "solo_1_acierta": b, "solo_2_acierta": c,
                  "discordancias": b + c, "p_valor": round(p, 4),
                  "significativo_0.05": p < 0.05})

mc = pd.DataFrame(filas)
mc.to_csv(OUT / "mcnemar_estabilidad.csv", index=False)
print(mc.to_string(index=False))

     condicion_1        condicion_2  solo_1_acierta  solo_2_acierta  discordancias  p_valor  significativo_0.05
  CLIP|caption_2   CLIP-L|caption_2               5              32             37   0.0000                True
CLIP-L|caption_2 LongCLIP|caption_2              17               9             26   0.1686               False
CLIP-L|caption_3 LongCLIP|caption_3              13              14             27   1.0000               False
  CLIP|caption_2 LongCLIP|caption_3               9              32             41   0.0004                True


### Lectura del resultado

Este es el hallazgo central del trabajo, y **contradice la expectativa inicial**.

El efecto de contexto largo —la razón de ser de LongCLIP— no se distingue del
ruido. Con discordancias casi parejas, el efecto está cerca de cero: más muestra lo
estimaría con más precisión *como cero*, no lo volvería significativo.

Un resultado negativo medido con rigor es un hallazgo legítimo. Lo que no sería
defendible es afirmar que LongCLIP gana por su ventana de contexto sin esta
verificación.

## 2.6 Truncación y su relación con el rendimiento (8.2.3)

Aquí está la razón por la que "caption_2 vs caption_3" no significa lo mismo en
todos los modelos.

Con 77 tokens, el caption largo que ve CLIP-L **no es el que está en el
manifiesto**: son sus primeros 77 tokens. Así que esa comparación no contrasta
corto contra largo, sino corto contra *largo recortado*.

In [8]:
# La auditoría de tokens se recalcula sobre los ítems de la fuente activa, no se
# lee de un CSV previo: si la fuente son las tres ediciones, la auditoría del
# Cuaderno14 solo cubriría 40 de los 95 y las tasas saldrían sesgadas.
def auditar_tokens(image_ids):
    man = pd.read_csv(MULTI / "manifest_multiedicion.csv")
    man = man[man["image_id"].isin(image_ids)]

    try:
        from transformers import CLIPTokenizerFast
        tok = CLIPTokenizerFast.from_pretrained(MODEL_SPECS["CLIP"]["checkpoint"])
        def n_tokens(t):
            return len(tok(str(t), truncation=False)["input_ids"])
        fuente_tok = "CLIPTokenizerFast"
    except Exception as e:
        # aproximación por palabras si transformers no está disponible; sirve
        # para el orden de magnitud pero no para cifras exactas
        def n_tokens(t):
            return int(len(str(t).split()) * 1.4) + 2
        fuente_tok = f"aproximación por palabras ({type(e).__name__})"

    filas = []
    for _, r in man.iterrows():
        for cap in CAPTIONS:
            n = n_tokens(r[cap])
            fila = {"image_id": r["image_id"], "caption": cap}
            for m, sp in MODEL_SPECS.items():
                fila[f"tokens_{m}"] = n
                fila[f"truncado_{m}"] = n > sp["max_tokens"]
            filas.append(fila)
    print(f"tokenización: {fuente_tok} · {len(man)} gráficos")
    return pd.DataFrame(filas)


aud = auditar_tokens(casos["image_id"])
aud.to_csv(OUT / "auditoria_truncacion.csv", index=False)

filas = []

for m in MODELOS:
    for c in CAPTIONS:
        s = aud[aud.caption == c]
        trunc = s[f"truncado_{m}"].astype(bool)

        sub = s[["image_id"]].copy()
        sub["truncado"] = trunc.values
        sub = sub.merge(casos[["image_id", f"rank_{m}_{c}"]], on="image_id")
        sub["acierto"] = sub[f"rank_{m}_{c}"] == 1

        r_t = sub.loc[sub.truncado, "acierto"].mean() if sub.truncado.any() else np.nan
        r_n = sub.loc[~sub.truncado, "acierto"].mean() if (~sub.truncado).any() else np.nan

        filas.append({"modelo": m, "caption": c,
                      "limite_tokens": MODEL_SPECS[m]["max_tokens"],
                      "pct_truncado": round(100 * trunc.mean(), 1),
                      "tokens_mediana": int(s[f"tokens_{m}"].median()),
                      "R@1_truncados": round(r_t, 3) if r_t == r_t else None,
                      "n_trunc": int(trunc.sum()),
                      "R@1_no_truncados": round(r_n, 3) if r_n == r_n else None,
                      "n_no_trunc": int((~trunc).sum())})

trunc_df = pd.DataFrame(filas)
trunc_df.to_csv(OUT / "resumen_truncacion.csv", index=False)
print(trunc_df.to_string(index=False))

/tf/work/torch_gpu_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Token indices sequence length is longer than the specified maximum sequence length for this model (130 > 77). Running this sequence through the model will result in indexing errors


tokenización: CLIPTokenizerFast · 95 gráficos
  modelo   caption  limite_tokens  pct_truncado  tokens_mediana  R@1_truncados  n_trunc  R@1_no_truncados  n_no_trunc
    CLIP caption_2             77           3.2              18          0.667        3             0.391          92
    CLIP caption_3             77          89.5             118          0.459       85             0.600          10
    CLIP caption_4             77          88.4             212          0.464       84             0.636          11
  CLIP-L caption_2             77           3.2              18          0.667        3             0.685          92
  CLIP-L caption_3             77          89.5             118          0.612       85             0.800          10
  CLIP-L caption_4             77          88.4             212          0.619       84             0.727          11
LongCLIP caption_2            248           0.0              18            NaN        0             0.600          95
LongCLIP c

## 2.7 Casos donde el título no basta (8.2.5)

`caption_2` es solo el título del gráfico; `caption_3` y `caption_4` agregan la
oración del informe con cifras. Si los modelos fallan con el título y aciertan al
añadir datos, entonces **el título no identifica el gráfico**: la respuesta depende
de la información numérica.

In [9]:
clasif = casos[["image_id", "chart_id"]].copy()
for c in CAPTIONS:
    clasif[f"aciertos_{c}"] = sum((casos[f"rank_{m}_{c}"] == 1).astype(int)
                                  for m in MODELOS)

def clasificar(r):
    solo_titulo = r["aciertos_caption_2"]
    con_datos = r["aciertos_caption_3"] + r["aciertos_caption_4"]
    if solo_titulo == 0 and con_datos >= 3:
        return "titulo_insuficiente"
    if solo_titulo == 0 and con_datos == 0:
        return "falla_total"
    if solo_titulo == 3 and con_datos == 6:
        return "acierto_total"
    return "mixto"

clasif["categoria"] = clasif.apply(clasificar, axis=1)
clasif.to_csv(OUT / "casos_titulo_insuficiente.csv", index=False)

print(clasif["categoria"].value_counts().to_string())
print("\nEl título no basta en:")
print(clasif[clasif.categoria == "titulo_insuficiente"].to_string(index=False))
print("\nNingún modelo los recupera nunca (negativos difíciles):")
print(clasif[clasif.categoria == "falla_total"][["image_id", "chart_id"]].to_string(index=False))

mixto                  66
acierto_total          17
falla_total             8
titulo_insuficiente     4

El título no basta en:
      image_id chart_id  aciertos_caption_2  aciertos_caption_3  aciertos_caption_4           categoria
2021-1_img_011    I.B.4                   0                   2                   1 titulo_insuficiente
       img_003     II.4                   0                   1                   2 titulo_insuficiente
       img_016   IV.A.2                   0                   2                   1 titulo_insuficiente
       img_036   VI.C.1                   0                   1                   2 titulo_insuficiente

Ningún modelo los recupera nunca (negativos difíciles):
      image_id chart_id
2021-1_img_010    I.B.3
2021-1_img_023      V.3
2024-2_img_006    I.A.7
2024-2_img_010    I.B.2
2024-2_img_024  III.B.4
2024-2_img_025  III.B.5
       img_019   IV.A.5
       img_024   IV.B.4


## 2.8 Recall@K con varios captions correctos (8.5.4)

Implementación de la tarea de código del examen. Tres detalles que la hacen
correcta y que conviene poder explicar:

1. **`argsort` ordena ascendente.** Tomar `argsort(similarity)[:, :k]` devuelve los
   *menos* similares. Hay que negar la matriz: `argsort(-similarity)`.
2. **K se acota** al número de columnas disponibles.
3. **Índices positivos duplicados se colapsan**, para que un mismo caption repetido
   no cuente dos veces.

In [10]:
def recall_at_k(similarity, positives, k):
    """Recall@K imagen->texto admitiendo varios captions correctos por imagen.

    similarity : matriz (n_imagenes x n_textos); mayor score = más similar
    positives  : positives[i] = índices de texto válidos para la imagen i
    """
    sim = np.asarray(similarity, dtype=float)
    n_img, n_txt = sim.shape
    if len(positives) != n_img:
        raise ValueError(f"positives tiene {len(positives)} filas y la matriz {n_img}")
    k = max(1, min(int(k), n_txt))

    orden = np.argsort(-sim, axis=1)[:, :k]      # descendente
    aciertos, duplicados = [], 0
    for i, pos in enumerate(positives):
        unicos = list(dict.fromkeys(pos))        # conserva orden, quita repetidos
        duplicados += len(pos) - len(unicos)
        if not unicos:
            raise ValueError(f"la imagen {i} no tiene captions correctos")
        if any(p < 0 or p >= n_txt for p in unicos):
            raise IndexError(f"índice fuera de rango en la imagen {i}")
        aciertos.append(any(p in orden[i] for p in unicos))

    if duplicados:
        print(f"  aviso: {duplicados} índices positivos duplicados ignorados")
    return float(np.mean(aciertos))


# verificación
sim = np.array([[0.9, 0.1, 0.2], [0.1, 0.8, 0.3], [0.2, 0.3, 0.7]])
print("caso perfecto      R@1:", recall_at_k(sim, [[0], [1], [2]], 1))
print("varios correctos   R@1:", recall_at_k(sim, [[0, 1], [1], [0, 2]], 1))
print("con duplicados     R@1:", recall_at_k(sim, [[0, 0, 0], [1, 1], [2]], 1))
print("K mayor que n      R@9:", recall_at_k(sim, [[0], [1], [2]], 9))
print()
print("argsort(sim)  ->", np.argsort(sim, axis=1)[:, 0], " (los PEORES: el error típico)")
print("argsort(-sim) ->", np.argsort(-sim, axis=1)[:, 0], " (los MEJORES: correcto)")

caso perfecto      R@1: 1.0
varios correctos   R@1: 1.0
  aviso: 3 índices positivos duplicados ignorados
con duplicados     R@1: 1.0
K mayor que n      R@9: 1.0

argsort(sim)  -> [1 0 0]  (los PEORES: el error típico)
argsort(-sim) -> [0 1 2]  (los MEJORES: correcto)


## 2.9 Casos analizados (requisito del repositorio)

La sección 3.2 del examen pide documentar dos aciertos, dos errores y un caso
ambiguo. Se seleccionan a partir de la clasificación anterior, para que la
elección sea reproducible y no un muestreo a dedo.

In [11]:
# Los casos analizados corresponden a la edición 2026-1, que es la evaluada en
# results/. Se toma del manifiesto combinado filtrando esa edición.
# el manifiesto cubre las tres ediciones; se filtra a los ítems de la fuente
man = pd.read_csv(MULTI / "manifest_multiedicion.csv")
man = man[man["image_id"].isin(casos["image_id"])]
clas = clasif.merge(man[["image_id", "caption_2", "source_page_pdf"]], on="image_id")

NOTAS = {
    "acierto_total": ("Los tres modelos lo recuperan con cualquiera de los tres captions: "
                      "el título describe un contenido visualmente distintivo."),
    "falla_total": ("Ningún modelo lo recupera con ningún caption. Comparte estructura "
                    "visual y vocabulario con otros gráficos del mismo capítulo, de modo "
                    "que el texto no lo discrimina: es un negativo difícil del propio pool."),
    "titulo_insuficiente": ("Con solo el título ningún modelo acierta; al agregar la oración "
                            "del informe con cifras, sí. Evidencia de que el título no basta "
                            "y la identificación depende de los datos."),
}
TIPO = {"acierto_total": "acierto", "falla_total": "error",
        "titulo_insuficiente": "ambiguo"}

sel = pd.concat([
    clas[clas.categoria == "acierto_total"].head(2),
    clas[clas.categoria == "falla_total"].head(2),
    clas[clas.categoria == "titulo_insuficiente"].head(1),
])

filas = []
for _, r in sel.iterrows():
    fila = {"image_id": r.image_id, "chart_id": r.chart_id,
            "tipo_caso": TIPO[r.categoria], "categoria": r.categoria,
            "caption_2": r.caption_2, "pagina_pdf": r.source_page_pdf,
            "aciertos_caption_2": r.aciertos_caption_2,
            "aciertos_caption_3": r.aciertos_caption_3,
            "aciertos_caption_4": r.aciertos_caption_4,
            "observacion": NOTAS[r.categoria]}
    for m in MODELOS:
        fila[f"rank_{m}_caption_2"] = int(
            casos.loc[casos.image_id == r.image_id, f"rank_{m}_caption_2"].iloc[0])
    filas.append(fila)

analizados = pd.DataFrame(filas)
analizados.to_csv(OUT / "casos_analizados.csv", index=False)
print(analizados[["image_id", "chart_id", "tipo_caso",
                  "aciertos_caption_2", "aciertos_caption_3"]].to_string(index=False))

      image_id chart_id tipo_caso  aciertos_caption_2  aciertos_caption_3
2021-1_img_014   II.A.1   acierto                   3                   3
2021-1_img_017   II.B.2   acierto                   3                   3
2021-1_img_010    I.B.3     error                   0                   0
2021-1_img_023      V.3     error                   0                   0
2021-1_img_011    I.B.4   ambiguo                   0                   2


## 2.10 Configuración experimental

Registro de todo lo necesario para reproducir el experimento: checkpoints,
límites de tokens, semilla y métrica principal. Es el archivo que permite a un
tercero repetir la corrida sin leer el código.

In [12]:
config = {
    "proyecto": "MCC225 Proyecto 2 - CLIP-B / CLIP-L / LongCLIP sobre gráficos SBS",
    "manifiesto": "manifest_multiedicion.csv",
    "n_graficos": int(len(casos)),
    "model_specs": MODEL_SPECS,
    "captions_evaluados": CAPTIONS,
    "metrica_principal": "i2t R@1 (la imagen recupera su caption correcto)",
    "metricas_secundarias": ["R@5", "MRR"],
    "semilla": SEED,
    "n_bootstrap": N_BOOT,
    "baseline_robustez": "captions desplazados una posición",
    "rutas": "relativas al directorio del proyecto",
    "amenaza_validez": ("los captions provienen del mismo informe que contiene el "
                        "gráfico, por lo que comparten vocabulario e incluso cifras "
                        "impresas en la imagen; esto probablemente infla los resultados"),
}

with open(OUT / "configuracion_experimental.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print(json.dumps(config, ensure_ascii=False, indent=2)[:600], "...")

{
  "proyecto": "MCC225 Proyecto 2 - CLIP-B / CLIP-L / LongCLIP sobre gráficos SBS",
  "manifiesto": "manifest_multiedicion.csv",
  "n_graficos": 95,
  "model_specs": {
    "CLIP": {
      "checkpoint": "openai/clip-vit-base-patch32",
      "max_tokens": 77,
      "encoder_visual": "ViT-B/32",
      "patch": 32
    },
    "CLIP-L": {
      "checkpoint": "openai/clip-vit-large-patch14",
      "max_tokens": 77,
      "encoder_visual": "ViT-L/14",
      "patch": 14
    },
    "LongCLIP": {
      "checkpoint": "zer0int/LongCLIP-GmP-ViT-L-14",
      "max_tokens": 248,
      "encoder_visual": "ViT-L ...
